# Notebook 1: Delta Memory Schema Setup
**agentic-civic-resolution-app**



 **Goal:** Create all persistent memory tables as Delta tables in `civicops.memory`

 ## Memory Architecture
 ```
 civicops.memory
├── complaint_history      — every complaint ever processed
├── recurring_complaints   — patterns at same location/type
├── escalation_context     — full escalation audit trail
└── sla_status_history     — every status/SLA change per ticket

**COMMAND**

 %pip install databricks-sdk psycopg2-binary sqlalchemy --quiet

 dbutils.library.restartPython()


## 0. Create Schema

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS civicops")
spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.memory")
print("✓ civicops.memory schema ready.")

## 1. complaint_history

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS civicops.memory.complaint_history (
    ticket_id                    STRING        NOT NULL,
    raw_complaint                STRING,
    category                     STRING,
    subcategory                  STRING,
    classification_confidence    DOUBLE,
    severity_score               INT,
    priority_label               STRING,
    sla_hours                    INT,
    sla_deadline                 TIMESTAMP,
    health_risk                  BOOLEAN,
    infrastructure_risk          BOOLEAN,
    affected_estimate            STRING,
    dept_code                    STRING,
    dept_name                    STRING,
    dept_contact                 STRING,
    officer_tier                 STRING,
    escalate                     BOOLEAN,
    escalation_dept              STRING,
    action_plan                  STRING,        -- JSON array stored as string
    field_visit_needed           BOOLEAN,
    notify_citizen               BOOLEAN,
    status                       STRING         DEFAULT 'OPEN',
    location_hint                STRING,
    processing_ms                INT,
    created_at                   TIMESTAMP      DEFAULT CURRENT_TIMESTAMP(),
    updated_at                   TIMESTAMP      DEFAULT CURRENT_TIMESTAMP(),
    resolved_at                  TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true' , 'delta.feature.allowColumnDefaults'='supported'
)
""")
print("✓ complaint_history created.")

## 2. recurring_complaints

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS civicops.memory.recurring_complaints (
    location_key        STRING      NOT NULL,
    complaint_type      STRING      NOT NULL,
    first_reported_at   TIMESTAMP,
    last_reported_at    TIMESTAMP,
    occurrence_count    INT         DEFAULT 1,
    avg_severity        DOUBLE,
    max_severity        INT,
    resolved_count      INT         DEFAULT 0,
    unresolved_count    INT         DEFAULT 0,
    ticket_ids          STRING,     -- JSON array of ticket_ids
    is_chronic          BOOLEAN     DEFAULT FALSE,
    chronic_flagged_at  TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported'
)
""")
print("✓ recurring_complaints created.")

## 3. escalation_context

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS civicops.memory.escalation_context (
    id                          BIGINT GENERATED ALWAYS AS IDENTITY,
    ticket_id                   STRING,
    escalated_at                TIMESTAMP   DEFAULT CURRENT_TIMESTAMP(),
    escalated_from_dept         STRING,
    escalated_to_dept           STRING,
    escalation_reason           STRING,
    severity_at_escalation      INT,
    health_risk                 BOOLEAN,
    infrastructure_risk         BOOLEAN,
    resolved_after_escalation   BOOLEAN     DEFAULT FALSE,
    resolution_time_hrs         DOUBLE,
    notes                       STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
""")
print("✓ escalation_context created.")

## 4. sla_status_history

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS civicops.memory.sla_status_history (
    id                          BIGINT GENERATED ALWAYS AS IDENTITY,
    ticket_id                   STRING,
    previous_status             STRING,
    new_status                  STRING,
    changed_at                  TIMESTAMP   DEFAULT CURRENT_TIMESTAMP(),
    changed_by                  STRING      DEFAULT 'system',
    sla_deadline                TIMESTAMP,
    sla_breached                BOOLEAN     DEFAULT FALSE,
    hours_remaining_at_change   DOUBLE,
    notes                       STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
""")
print("✓ sla_status_history created.")

## 5. Verify

In [0]:
tables = spark.sql("SHOW TABLES IN civicops.memory").collect()
print(f"{'Table':<30} {'Type'}")
print("-" * 45)
for t in tables:
    print(f"{t['tableName']:<30} Delta")

print(f"\n✓ {len(tables)} memory tables ready in civicops.memory")